# <font color="#418FDE" size="6.5" uppercase>**Boosted Trees**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Fit gradient boosting classifiers and regressors for supervised tasks. 
- Tune learning rate, estimator count, subsampling, and early stopping. 
- Use histogram gradient boosting features such as native categories, missingness, and monotonic constraints. 


## **1. Gradient Boosting Basics**

### **1.1. Gradient Boosting Classification**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_18/Lecture_A/image_01_01.jpg?v=1784011152" width="250">



>* Sequential trees improve classification mistakes
>* Combined trees capture complex class patterns

>* Starts simple, improves with shallow trees
>* Predicts classes from scores or probabilities

>* Accurate, flexible models for structured data
>* Validate carefully to avoid overfitting costs



In [ ]:
#@title Python Code - Gradient Boosting Classification

# This example trains a gradient boosting classifier.
# Shallow trees improve predictions step by step.
# Test accuracy shows generalization on unseen flowers.

import sklearn
from sklearn.datasets import load_iris
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Load a small built-in multiclass classification dataset.
iris = load_iris()
features = iris.data
target = iris.target

# Validate the basic shape before modeling.
if features.shape[0] != target.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split data so evaluation uses unseen examples.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.3,
    stratify=target,
    random_state=42,
)

# Fit many small trees sequentially with gentle updates.
model = GradientBoostingClassifier(
    n_estimators=60,
    learning_rate=0.08,
    max_depth=2,
    random_state=42,
)

model.fit(X_train, y_train)

# Predict classes and class probabilities for test flowers.
predicted_classes = model.predict(X_test)
predicted_probabilities = model.predict_proba(X_test)

# Summarize performance and one probability-based prediction.
accuracy = accuracy_score(y_test, predicted_classes)
first_name = iris.target_names[predicted_classes[0]]
first_confidence = predicted_probabilities[0].max()

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Test accuracy: {accuracy:.3f}")
print(f"First test prediction: {first_name}, confidence {first_confidence:.3f}")



### **1.2. Gradient Boosting Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_18/Lecture_A/image_01_02.jpg?v=1784011154" width="250">



>* Predict continuous values using boosted decision trees
>* Sequential trees correct remaining prediction errors

>* Small trees add sequential prediction adjustments
>* Later trees capture complex nonlinear patterns

>* Simple trees combine to model complex patterns
>* Control growth to avoid memorizing noise



In [ ]:
#@title Python Code - Gradient Boosting Regression

# Fit gradient boosting for continuous prediction.
# Sequential trees reduce remaining regression errors.
# Compare predictions with held-out target values.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_diabetes
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

# Load a small packaged regression dataset.
diabetes = load_diabetes()
features = diabetes.data
target = diabetes.target

# Validate the basic feature and target shapes.
if features.shape[0] != target.shape[0]:
    raise ValueError("Feature rows must match target values.")

# Split data before fitting the model.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, random_state=42
)

# Fit many shallow trees sequentially.
model = GradientBoostingRegressor(
    n_estimators=80, learning_rate=0.05, max_depth=2, random_state=42
)
model.fit(X_train, y_train)

# Predict unseen examples and measure average error.
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)

print("scikit-learn version:", sklearn.__version__)
print("Test mean absolute error:", round(mae, 2))
print("First prediction versus actual:", round(predictions[0], 1), round(y_test[0], 1))

# Plot predicted values against actual values.
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(y_test, predictions, alpha=0.75, label="Test patients")
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], label="Perfect prediction")
ax.set_title("Gradient Boosting Regression Predictions")
ax.set_xlabel("Actual disease progression score")
ax.set_ylabel("Predicted disease progression score")
ax.legend()
plt.show()



### **1.3. Boosting Losses**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_18/Lecture_A/image_01_03.jpg?v=1784011149" width="250">



>* Losses define mistakes for each boosting step
>* They guide regression errors and classification confidence

>* Loss choice shapes regression error sensitivity
>* Match loss to outliers and real costs

>* Losses shape class probabilities and confidence
>* Focus hard cases without overfitting noise



In [ ]:
#@title Python Code - Boosting Losses

# This example compares two boosting regression losses.
# Loss choice changes how outliers influence learning.
# The printed scores reveal robust loss behavior.

import numpy as np
import sklearn
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

# Create a small regression dataset with one clear pattern.
rng = np.random.default_rng(42)
x_values = np.linspace(0, 10, 160)
noise = rng.normal(0, 0.4, size=x_values.shape)

# Add a few large outliers to challenge squared error.
y_values = 2.0 * x_values + noise
outlier_positions = np.arange(0, 160, 20)
y_values[outlier_positions] = y_values[outlier_positions] + 12

# Scikit-learn expects a two-dimensional feature matrix.
features = x_values.reshape(-1, 1)
target = y_values

# Validate the simple shape assumption before modeling.
if features.shape[0] != target.shape[0]:
    raise ValueError("Features and target must have matching rows.")

# Split once so both losses face the same test examples.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.3, random_state=42
)

# Squared error gives very large mistakes strong influence.
squared_model = GradientBoostingRegressor(
    loss="squared_error", n_estimators=80, learning_rate=0.08, random_state=42
)
squared_model.fit(X_train, y_train)

# Absolute error is usually less pulled by extreme targets.
absolute_model = GradientBoostingRegressor(
    loss="absolute_error", n_estimators=80, learning_rate=0.08, random_state=42
)
absolute_model.fit(X_train, y_train)

# Compare both models using the same easy-to-read metric.
squared_mae = mean_absolute_error(y_test, squared_model.predict(X_test))
absolute_mae = mean_absolute_error(y_test, absolute_model.predict(X_test))

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Test MAE with squared_error loss: {squared_mae:.2f}")
print(f"Test MAE with absolute_error loss: {absolute_mae:.2f}")
print("Lower MAE means better typical prediction on this noisy test set.")



## **2. Boosting Control**

### **2.1. Learning Rate Tradeoff**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_18/Lecture_A/image_02_01.jpg?v=1784011164" width="250">



>* High rates learn fast but risk noise
>* Low rates learn steadily with more trees

>* Lower rates need more trees
>* Higher rates learn faster but overreact

>* Balance learning speed, accuracy, and stability
>* Use validation and early stopping to generalize



In [ ]:
#@title Python Code - Learning Rate Tradeoff

# Compare learning rates in gradient boosting.
# Smaller steps usually need more trees.
# Validation curves reveal the tradeoff clearly.

import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split

# Create a small deterministic classification dataset.
features, target = make_classification(
    n_samples=1200,
    n_features=12,
    n_informative=6,
    n_redundant=2,
    random_state=42,
)

# Split data before fitting any model.
train_features, valid_features, train_target, valid_target = train_test_split(
    features, target, test_size=0.30, stratify=target, random_state=42
)

# Validate the split before modeling.
if train_features.shape[0] == 0 or valid_features.shape[0] == 0:
    raise ValueError("Training and validation sets must be nonempty.")

print("scikit-learn version:", sklearn.__version__)

learning_rates = [0.03, 0.10, 0.30]
colors = ["tab:blue", "tab:green", "tab:red"]

# Create one axes for comparing validation loss curves.
fig, ax = plt.subplots(figsize=(8, 5))

for learning_rate, color in zip(learning_rates, colors):
    model = GradientBoostingClassifier(
        learning_rate=learning_rate,
        n_estimators=120,
        random_state=42,
    )

    model.fit(train_features, train_target)
    validation_losses = []

    for probabilities in model.staged_predict_proba(valid_features):
        loss = log_loss(valid_target, probabilities)
        validation_losses.append(loss)

    best_loss = min(validation_losses)
    best_tree_count = validation_losses.index(best_loss) + 1
    label = f"rate={learning_rate}, best trees={best_tree_count}"

    ax.plot(range(1, 121), validation_losses, color=color, label=label)
    print(f"rate {learning_rate}: best validation log loss {best_loss:.3f}")

ax.set_title("Learning Rate Tradeoff in Gradient Boosting")
ax.set_xlabel("Number of trees added")
ax.set_ylabel("Validation log loss, lower is better")
ax.legend()

plt.show()



### **2.2. Stochastic Boosting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_18/Lecture_A/image_02_02.jpg?v=1784011162" width="250">



>* Train trees on random data subsets
>* Reduce overfitting to unusual training cases

>* Subsampling controls randomness and variance
>* Too little data makes learning unstable

>* Tune subsampling with learning rate and estimators
>* Use early stopping carefully amid random fluctuations



In [ ]:
#@title Python Code - Stochastic Boosting

# Compare full-data and stochastic gradient boosting.
# Subsampling adds controlled randomness during boosting.
# Validation scores reveal the regularization effect.

import sklearn
from sklearn.datasets import make_regression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

# Create a small noisy regression dataset.
features, target = make_regression(
    n_samples=1200,
    n_features=12,
    noise=25.0,
    random_state=42,
)

# Split once so both models face the same test data.
train_features, test_features, train_target, test_target = train_test_split(
    features,
    target,
    test_size=0.25,
    random_state=42,
)

# Validate the expected two-dimensional feature table.
if train_features.shape[1] != 12:
    raise ValueError("Expected 12 input features for this lesson.")

# Fit standard boosting using every training row per tree.
full_model = GradientBoostingRegressor(
    n_estimators=180,
    learning_rate=0.05,
    subsample=1.0,
    random_state=42,
)
full_model.fit(train_features, train_target)

# Fit stochastic boosting using only part of the rows.
stochastic_model = GradientBoostingRegressor(
    n_estimators=180,
    learning_rate=0.05,
    subsample=0.6,
    random_state=42,
)
stochastic_model.fit(train_features, train_target)

# Compare training and test errors for both settings.
full_train_mae = mean_absolute_error(
    train_target,
    full_model.predict(train_features),
)
full_test_mae = mean_absolute_error(test_target, full_model.predict(test_features))

stochastic_train_mae = mean_absolute_error(
    train_target,
    stochastic_model.predict(train_features),
)
stochastic_test_mae = mean_absolute_error(
    test_target,
    stochastic_model.predict(test_features),
)

# Print concise results that focus on subsampling.
print("scikit-learn version:", sklearn.__version__)
print("Full-data boosting MAE train/test:", round(full_train_mae, 1), round(full_test_mae, 1))
print("Stochastic boosting MAE train/test:", round(stochastic_train_mae, 1), round(stochastic_test_mae, 1))
print("Subsample 0.6 trains each tree on 60% of rows.")



### **2.3. Early Stopping**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_18/Lecture_A/image_02_03.jpg?v=1784011166" width="250">



>* Stop boosting when validation gains stall
>* Prevent overfitting to training noise

>* Use patience for fluctuating validation scores
>* Stop efficiently while allowing further improvement

>* Learning rate affects needed tree count
>* Validation stops growth before overfitting



In [ ]:
#@title Python Code - Early Stopping

# Demonstrate early stopping in gradient boosting.
# Track validation accuracy after each tree.
# Show where extra trees stop helping.

import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Create a small deterministic classification dataset.
features, target = make_classification(
    n_samples=1200,
    n_features=12,
    n_informative=6,
    n_redundant=2,
    random_state=42,
)

# Split once so validation data stays unseen during fitting.
train_features, valid_features, train_target, valid_target = train_test_split(
    features,
    target,
    test_size=0.30,
    stratify=target,
    random_state=42,
)

# Validate the split before training the model.
if train_features.shape[0] == 0 or valid_features.shape[0] == 0:
    raise ValueError("Training and validation sets must both contain rows.")

# Fit many trees, then inspect validation accuracy after each stage.
model = GradientBoostingClassifier(
    n_estimators=120,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42,
)
model.fit(train_features, train_target)

# Store validation accuracy for every boosting stage.
validation_scores = []
for predictions in model.staged_predict(valid_features):
    score = accuracy_score(valid_target, predictions)
    validation_scores.append(score)

# Find the best stage and a simple patience-based stopping point.
best_index = int(max(range(len(validation_scores)), key=validation_scores.__getitem__))
best_tree_count = best_index + 1
best_score = validation_scores[best_index]
patience = 10

# Simulate early stopping with patience after the best score.
early_stop_tree_count = min(best_tree_count + patience, len(validation_scores))
final_score = validation_scores[-1]

# Print concise results that connect trees to validation performance.
print("scikit-learn version:", sklearn.__version__)
print("Best validation accuracy:", round(best_score, 3))
print("Best tree count:", best_tree_count)
print("Patience-based stop count:", early_stop_tree_count)
print("Accuracy after all 120 trees:", round(final_score, 3))

# Plot validation accuracy to reveal the plateau.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(validation_scores) + 1), validation_scores, label="Validation")
ax.axvline(best_tree_count, color="green", linestyle="--", label="Best score")
ax.axvline(early_stop_tree_count, color="red", linestyle=":", label="Stop point")
ax.set_title("Early stopping watches validation accuracy")
ax.set_xlabel("Number of boosted trees")
ax.set_ylabel("Validation accuracy")
ax.legend()
plt.show()



## **3. Histogram Boosting**

### **3.1. HistGradientBoosting Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_18/Lecture_A/image_03_01.jpg?v=1784011156" width="250">



>* Bins speed up boosted tree split searches
>* Preserves nonlinear patterns and feature interactions

>* Works well for structured prediction tasks
>* Captures nonlinear patterns efficiently at scale

>* Handles missing values and categorical features
>* Supports constraints for domain-aware predictions



In [ ]:
#@title Python Code - HistGradientBoosting Basics

# This example trains histogram gradient boosting.
# It demonstrates missing values and monotonic constraints.
# The output compares constrained and unconstrained predictions.

import numpy as np
import sklearn
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

# Create a small deterministic regression dataset.
rng = np.random.default_rng(42)
n_samples = 800
income = rng.uniform(20, 120, n_samples)

# Add a second feature with some missing values.
debt_ratio = rng.uniform(0.05, 0.75, n_samples)
missing_mask = rng.random(n_samples) < 0.12
debt_ratio[missing_mask] = np.nan

# Higher income should generally increase the target.
noise = rng.normal(0, 8, n_samples)
risk_score = 0.7 * income - 35 * np.nan_to_num(debt_ratio, nan=0.4) + noise

# Combine features into one training matrix.
features = np.column_stack([income, debt_ratio])
if features.shape != (n_samples, 2):
    raise ValueError("Expected two feature columns.")

# Split data before fitting the model.
X_train, X_test, y_train, y_test = train_test_split(
    features, risk_score, test_size=0.25, random_state=42
)

# Fit a basic histogram gradient boosting regressor.
basic_model = HistGradientBoostingRegressor(
    max_iter=80, learning_rate=0.08, random_state=42
)
basic_model.fit(X_train, y_train)

# Fit a model constrained to increase with income.
constrained_model = HistGradientBoostingRegressor(
    max_iter=80, learning_rate=0.08, monotonic_cst=[1, 0], random_state=42
)
constrained_model.fit(X_train, y_train)

# Evaluate both models on held-out data.
basic_mae = mean_absolute_error(y_test, basic_model.predict(X_test))
constrained_mae = mean_absolute_error(y_test, constrained_model.predict(X_test))

# Test predictions as income rises while debt is missing.
income_grid = np.array([30, 50, 70, 90, 110], dtype=float)
missing_debt_grid = np.full_like(income_grid, np.nan)
grid = np.column_stack([income_grid, missing_debt_grid])

# Print concise results for beginners.
print("scikit-learn version:", sklearn.__version__)
print("Basic model MAE:", round(basic_mae, 2))
print("Constrained model MAE:", round(constrained_mae, 2))
print("Income grid:", income_grid.astype(int).tolist())
print("Constrained predictions:", np.round(constrained_model.predict(grid), 1).tolist())



### **3.2. Native Categories**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_18/Lecture_A/image_03_02.jpg?v=1784011160" width="250">



>* Handle categories without one-hot encoding
>* Avoid false numeric ordering of labels

>* Learns predictive groupings among categories
>* Keeps pipelines simpler and categories explicit

>* Clean categories and combine rare values
>* Plan for unseen categories using domain knowledge



In [ ]:
#@title Python Code - Native Categories

# Demonstrate native categories in histogram boosting.
# Categorical labels stay distinct, not ordered.
# The model predicts prices from mixed features.

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder

# Build a small housing dataset with categorical neighborhoods.
rng = np.random.default_rng(42)
n_samples = 240
neighborhoods = np.array(["central", "suburb", "riverside", "industrial"])

# Create numeric and categorical predictors.
area = rng.normal(120, 25, n_samples).clip(60, 190)
age = rng.integers(0, 45, n_samples)
chosen_neighborhood = rng.choice(neighborhoods, n_samples)

# Give each category a different price effect.
category_effects = {
    "central": 80,
    "suburb": 35,
    "riverside": 60,
    "industrial": 10,
}

# Combine effects into a regression target.
effects = np.array([category_effects[name] for name in chosen_neighborhood])
noise = rng.normal(0, 8, n_samples)
price = 90 + 2.1 * area - 1.2 * age + effects + noise

# Store the category using pandas categorical dtype.
data = pd.DataFrame(
    {"area": area, "age": age, "neighborhood": chosen_neighborhood}
)
data["neighborhood"] = data["neighborhood"].astype("category")

# Split before fitting to keep evaluation honest.
X_train, X_test, y_train, y_test = train_test_split(
    data, price, test_size=0.25, random_state=42
)

# Validate that the categorical column is still categorical.
if str(X_train["neighborhood"].dtype) != "category":
    raise ValueError("The neighborhood column must be categorical.")

# Tell the model which column is categorical.
model = HistGradientBoostingRegressor(
    categorical_features=["neighborhood"], random_state=42, max_iter=80
)
model.fit(X_train, y_train)

# Compare with arbitrary integer codes to show the teaching point.
code_transformer = ColumnTransformer(
    [("category_codes", OrdinalEncoder(), ["neighborhood"])],
    remainder="passthrough",
)

# This pipeline treats category codes as ordered numbers.
ordered_code_model = make_pipeline(
    code_transformer,
    HistGradientBoostingRegressor(random_state=42, max_iter=80),
)
ordered_code_model.fit(X_train, y_train)

# Evaluate both approaches on the same test rows.
native_mae = mean_absolute_error(y_test, model.predict(X_test))
ordered_mae = mean_absolute_error(y_test, ordered_code_model.predict(X_test))

print("scikit-learn version:", sklearn.__version__)
print("Native categorical MAE:", round(native_mae, 2))
print("Arbitrary ordered-code MAE:", round(ordered_mae, 2))
print("Categorical feature used directly:", "neighborhood")



### **3.3. Monotonic Constraints**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_18/Lecture_A/image_03_03.jpg?v=1784011158" width="250">



>* Guide predictions using expected feature directions
>* Reduce noisy patterns while keeping flexibility

>* Constrain only features with clear direction
>* Leave non-monotonic features unconstrained

>* Improve trust, stability, and interpretability
>* Validate constraints with data and experts



In [ ]:
#@title Python Code - Monotonic Constraints

# Demonstrate monotonic constraints in histogram boosting.
# Compare constrained and unconstrained model behavior.
# Predictions should respect the chosen direction.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.ensemble import HistGradientBoostingRegressor

# Create a small noisy regression dataset.
rng = np.random.default_rng(42)
income = rng.uniform(20, 120, size=500)
noise = rng.normal(0, 18, size=500)
price = 50 + 2.0 * income + noise

# Add a misleading noisy dip in the training data.
dip_mask = (income > 70) & (income < 85)
price[dip_mask] = price[dip_mask] - 70
X = income.reshape(-1, 1)

# Validate the simple one-feature training shape.
if X.shape != (500, 1):
    raise ValueError("Expected 500 rows and one feature.")

# Fit the same model with and without a constraint.
free_model = HistGradientBoostingRegressor(random_state=42, max_iter=80)
free_model.fit(X, price)
constrained_model = HistGradientBoostingRegressor(
    random_state=42, max_iter=80, monotonic_cst=[1]
)
constrained_model.fit(X, price)

# Predict across increasing income values.
income_grid = np.linspace(20, 120, 120).reshape(-1, 1)
free_predictions = free_model.predict(income_grid)
constrained_predictions = constrained_model.predict(income_grid)

# Count downward steps as monotonicity violations.
free_drops = int(np.sum(np.diff(free_predictions) < -1e-9))
constrained_drops = int(np.sum(np.diff(constrained_predictions) < -1e-9))
print("scikit-learn version:", sklearn.__version__)
print("Unconstrained downward steps:", free_drops)
print("Constrained downward steps:", constrained_drops)

# Plot both prediction curves for visual comparison.
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(income, price, s=12, alpha=0.25, label="training data")
ax.plot(income_grid.ravel(), free_predictions, label="unconstrained")
ax.plot(income_grid.ravel(), constrained_predictions, label="monotonic +1")
ax.set_title("Monotonic constraint prevents downward prediction steps")
ax.set_xlabel("Income feature")
ax.set_ylabel("Predicted target")
ax.legend()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Boosted Trees**</font>


In this lecture, you learned to:
- Fit gradient boosting classifiers and regressors for supervised tasks. 
- Tune learning rate, estimator count, subsampling, and early stopping. 
- Use histogram gradient boosting features such as native categories, missingness, and monotonic constraints. 

In the next Lecture (Lecture B), we will go over 'Meta Ensembles'